24.11.2026

## 7

Загрузите orders.csv. Создайте таблицу df1, в которой:

• client_clean - имя клиента без лишних пробелов в начале/конце и в формате Title Case

• city_clean - город аналогично

• category_clean - категория без пробелов и в нижнем регистре

• product_clean - товар без пробелов и в верхнем регистре

• revenue - выручка по формуле qty * price * (1 - discount), округлённая до 2 знаков

In [1]:
import pandas as pd

df1 = pd.read_csv("orders.csv")
df1["client_clean"] = df1["client"].str.strip().str.title()
df1["city_clean"] = df1["city"].str.strip().str.title()
df1["category_clean"] = df1["category"].str.replace(" ", "", regex=False).str.lower()
df1["product_clean"] = df1["product"].str.replace(" ", "", regex=False).str.upper()
df1["revenue"] = (df1["qty"] * df1["price"] * (1 - df1["discount"])).round(2)

print(df1[["client_clean", "city_clean", "category_clean", "product_clean", "revenue"]].to_string(index=False))

      client_clean       city_clean category_clean product_clean  revenue
       Ivan Petrov           Moscow    electronics        LAPTOP 135000.0
       Ivan Petrov           Moscow    electronics        LAPTOP  68400.0
     Anna Smirnova Saint Petersburg           home         CHAIR  13500.0
      Pavel Ivanov            Kazan    electronics       MONITOR  15300.0
    Maria Sokolova      Novosibirsk         office          DESK  19200.0
     Sergey Volkov    Yekaterinburg    electronics      KEYBOARD  10000.0
     Elena Petrova           Moscow           home          SOFA  41250.0
    Dmitry Smirnov Saint Petersburg         office       PRINTER  13500.0
   Olga Kuznetsova            Kazan    electronics        TABLET  57000.0
     Anna Smirnova Saint Petersburg    electronics         PHONE  45000.0
     Anna Smirnova Saint Petersburg    electronics         PHONE  96000.0
      Igor Fedorov           Moscow           home          LAMP   7500.0
  Natalia Morozova      Novosibirsk   

## 8

На основе df1 создайте df2:

• добавьте revenue_rank — плотный ранг заказа по убыванию выручки (method="dense")

• отсортируйте все заказы по убыванию revenue, а при равенстве — по возрастанию order_date

• не используйте фильтрацию и head()

In [2]:
df2 = df1.copy()
df2["revenue_rank"] = df2["revenue"].rank(method="dense", ascending=False).astype(int)
df2 = df2.sort_values(["revenue", "order_date"], ascending=[False, True])

print(df2.to_string(index=False))

 order_id             client               city order_date      category        product  qty   price  discount    status  delivery_days        manager promo_code       client_clean       city_clean category_clean product_clean  revenue  revenue_rank
        1       Ivan Petrov             Moscow  2024-01-05  Electronics         laptop     2 75000.0      0.10      paid              3   Ivan Sidorov     SAVE10        Ivan Petrov           Moscow    electronics        LAPTOP 135000.0             1
       11     anna smirnova   saint petersburg  2024-02-14  electronics          Phone     2 48000.0      0.00       new              4 Petr Kuznetsov   NO_PROMO      Anna Smirnova Saint Petersburg    electronics         PHONE  96000.0             2
        2        ivan petrov             moscow 2024-01-05   electronics         LAPTOP    1 72000.0      0.05   shipped              5   Ivan Sidorov   NO_PROMO        Ivan Petrov           Moscow    electronics        LAPTOP  68400.0             3


## 9

На основе df2 создайте df3:

• преобразуйте order_date в datetime

• добавьте order_year, order_month, order_quarter

• добавьте order_weekday - название дня недели

• добавьте is_weekend - признак выходного дня

• добавьте delivery_date - дата заказа + delivery_days

• добавьте days_to_month_end - сколько дней осталось до конца месяца заказа

In [3]:
df3 = df2.copy()
df3["order_date"] = pd.to_datetime(df3["order_date"])
df3["order_year"] = df3["order_date"].dt.year
df3["order_month"] = df3["order_date"].dt.month
df3["order_quarter"] = df3["order_date"].dt.quarter
df3["order_weekday"] = df3["order_date"].dt.day_name()
df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5
df3["delivery_date"] = df3["order_date"] + pd.to_timedelta(df3["delivery_days"], unit="D")
df3["days_to_month_end"] = df3["order_date"].dt.days_in_month - df3["order_date"].dt.day

print(df3[["order_id", "order_date", "order_year", "order_month", "order_quarter", "order_weekday", "is_weekend", "delivery_date", "days_to_month_end"]].to_string(index=False))

 order_id order_date  order_year  order_month  order_quarter order_weekday  is_weekend delivery_date  days_to_month_end
        1 2024-01-05        2024            1              1        Friday       False    2024-01-08                 26
       11 2024-02-14        2024            2              1     Wednesday       False    2024-02-18                 15
        2 2024-01-05        2024            1              1        Friday       False    2024-01-10                 26
        9 2024-01-22        2024            1              1        Monday       False    2024-01-25                  9
       14 2024-02-22        2024            2              1      Thursday       False    2024-02-25                  7
       10 2024-02-14        2024            2              1     Wednesday       False    2024-02-16                 15
       30 2024-04-03        2024            4              2     Wednesday       False    2024-04-10                 27
        7 2024-01-18        2024        

C:\Users\sbtim\AppData\Local\Temp\ipykernel_29564\2299408867.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df3["is_weekend"] = df3["order_date"].dt.dayofweek >= 5


## 10

Считаем дублем комбинацию client_clean, order_date, product_clean.

На основе df3 создайте df4:

• добавьте duplicate_group_flag — True, если такая комбинация встречается более одного раза

• оставьте по одной строке на каждую комбинацию - строку с максимальной revenue

• после удаления дублей отсортируйте результат по order_id

In [4]:
df4 = df3.copy()
columns = ["client_clean", "order_date", "product_clean"]
df4["duplicate_group_flag"] = df4.duplicated(subset=columns, keep=False)
df4 = df4.sort_values("revenue", ascending=False)
df4 = df4.drop_duplicates(subset=columns)
df4 = df4.sort_values("order_id")

print(df4[["order_id", "client_clean", "order_date", "product_clean", "revenue", "duplicate_group_flag"]].to_string(index=False))
print("Удалено дублей:", len(df3) - len(df4))

 order_id       client_clean order_date product_clean  revenue  duplicate_group_flag
        1        Ivan Petrov 2024-01-05        LAPTOP 135000.0                  True
        3      Anna Smirnova 2024-01-07         CHAIR  13500.0                 False
        4       Pavel Ivanov 2024-01-10       MONITOR  15300.0                 False
        5     Maria Sokolova 2024-01-12          DESK  19200.0                 False
        6      Sergey Volkov 2024-01-15      KEYBOARD  10000.0                 False
        7      Elena Petrova 2024-01-18          SOFA  41250.0                 False
        8     Dmitry Smirnov 2024-01-20       PRINTER  13500.0                 False
        9    Olga Kuznetsova 2024-01-22        TABLET  57000.0                 False
       11      Anna Smirnova 2024-02-14         PHONE  96000.0                  True
       12       Igor Fedorov 2024-02-16          LAMP   7500.0                 False
       13   Natalia Morozova 2024-02-20      NOTEBOOK   2250.0   

## 11

На основе df4 создайте df5:

revenue_segment - категория выручки:

• low: [0, 1000);

• medium: [1000, 5000);

• high: [5000, 20000);

• vip: [20000, +∞);

status_ru - перевод статуса:

• new -> новый

• paid -> оплачен

• shipped -> отправлен

• cancelled -> отменён

• returned -> возврат

promo_used - True, если promo_code не равен NO_PROMO.

In [5]:
df5 = df4.copy()
df5["revenue_segment"] = pd.cut(df5["revenue"], bins=[0, 1000, 5000, 20000, float("inf")], labels=["low", "medium", "high", "vip"], right=False)

statuses = {
    "new": "новый",
    "paid": "оплачен",
    "shipped": "отправлен",
    "cancelled": "отменён",
    "returned": "возврат"
}
df5["status_ru"] = df5["status"].replace(statuses)
df5["promo_used"] = df5["promo_code"] != "NO_PROMO"

print(df5[["order_id", "revenue", "revenue_segment", "status_ru", "promo_used"]].to_string(index=False))

 order_id  revenue revenue_segment status_ru  promo_used
        1 135000.0             vip   оплачен        True
        3  13500.0            high     новый        True
        4  15300.0            high   оплачен        True
        5  19200.0            high отправлен       False
        6  10000.0            high   отменён       False
        7  41250.0             vip   оплачен        True
        8  13500.0            high   возврат       False
        9  57000.0             vip отправлен        True
       11  96000.0             vip     новый       False
       12   7500.0            high отправлен       False
       13   2250.0          medium   оплачен        True
       14  45500.0             vip   оплачен        True
       15  15300.0            high отправлен       False
       16  15960.0            high     новый        True
       17  19200.0            high   оплачен        True
       18  31500.0             vip   отменён       False
       19   6000.0            h

## 12

На основе df5 создайте df6:

• добавьте priority по правилам:

  - если status равен cancelled или returned -> archive
  - иначе если revenue >= 20000 и delivery_days <= 3 → «A»;
  - иначе если revenue >= 5000 -> «B»;
  - иначе -> «C»;

• добавьте manager_email - email вида имя.фамилия@shop.ru из manager в нижнем регистре, пробелы заменены на точки

• добавьте final_comment - строка вида "<client_clean>: <priority>, <revenue_segment>"

• сохраните итог в orders_final.csv

In [6]:
df6 = df5.copy()

def get_priority(row):
    if row["status"] in ["cancelled", "returned"]:
        return "archive"
    elif row["revenue"] >= 20000 and row["delivery_days"] <= 3:
        return "A"
    elif row["revenue"] >= 5000:
        return "B"
    else:
        return "C"

df6["priority"] = df6.apply(get_priority, axis=1)
df6["manager_email"] = df6["manager"].str.lower().str.replace(" ", ".", regex=False) + "@shop.ru"
df6["final_comment"] = df6["client_clean"] + ": " + df6["priority"] + ", " + df6["revenue_segment"].astype(str)

df6.to_csv("orders_final.csv", index=False, encoding="utf-8-sig")

print(df6[["order_id", "priority", "manager_email", "final_comment"]].to_string(index=False))
print("Файл orders_final.csv сохранён")

 order_id priority          manager_email                     final_comment
        1        A   ivan.sidorov@shop.ru               Ivan Petrov: A, vip
        3        B petr.kuznetsov@shop.ru            Anna Smirnova: B, high
        4        B   anna.volkova@shop.ru             Pavel Ivanov: B, high
        5        B  olga.morozova@shop.ru           Maria Sokolova: B, high
        6  archive   dmitry.orlov@shop.ru      Sergey Volkov: archive, high
        7        B   ivan.sidorov@shop.ru             Elena Petrova: B, vip
        8  archive petr.kuznetsov@shop.ru     Dmitry Smirnov: archive, high
        9        A   anna.volkova@shop.ru           Olga Kuznetsova: A, vip
       11        B petr.kuznetsov@shop.ru             Anna Smirnova: B, vip
       12        B   ivan.sidorov@shop.ru             Igor Fedorov: B, high
       13        C  olga.morozova@shop.ru       Natalia Morozova: C, medium
       14        A   dmitry.orlov@shop.ru              Alexey Popov: A, vip
       15   